<a href="https://colab.research.google.com/github/marcosvpm1708/Computa-o-de-baixo-desempenho/blob/main/Monte_Carlo_OMP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
%%writefile monte_carlo_omp.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <math.h>
#include <omp.h>

#define MAX_LINE_LEN 4096
#define NUM_NUMERICAL_COLS 16
#define NUM_CATEGORICAL_COLS 2

typedef struct {
    int ano;
    char uf[4];
    char cobrade[8];
    double numerical_data[NUM_NUMERICAL_COLS];
} DisasterRecord;

typedef struct {
    char** categories;
    double* probabilities;
    int count;
    int capacity;
} CategoricalStats;

typedef struct {
    double means[NUM_NUMERICAL_COLS];
    double std_devs[NUM_NUMERICAL_COLS];
} NumericalStats;

void load_and_split_data(const char* filename, DisasterRecord** train_data, long* train_count, DisasterRecord** valid_data, long* valid_count);
void calculate_stats_from_train(const DisasterRecord* train_data, long train_count, NumericalStats* num_stats, CategoricalStats cat_stats[NUM_CATEGORICAL_COLS]);
void run_simulation(long n_events, const NumericalStats* num_stats, const CategoricalStats cat_stats[NUM_CATEGORICAL_COLS], DisasterRecord* sim_data);
void compare_and_print_stats(const DisasterRecord* valid_data, long valid_count, const DisasterRecord* sim_data, long sim_count);
void save_results(const char* filename, const DisasterRecord* sim_data, long sim_count);
void cleanup(DisasterRecord* train_data, DisasterRecord* valid_data, DisasterRecord* sim_data, CategoricalStats cat_stats[NUM_CATEGORICAL_COLS]);
double generate_normal_sample(double mean, double std_dev, unsigned int* seed);

const char* INPUT_FILE = "/content/dataset_montecarlo_OMP.csv";
const char* OUTPUT_FILE = "montecarlo_OMP.csv";

int main() {
    double start_time = omp_get_wtime();
    printf("Iniciando processo de simulação de Monte Carlo.\n\n");

    DisasterRecord *train_data = NULL, *valid_data = NULL, *sim_data = NULL;
    long train_count = 0, valid_count = 0;

    printf("Lendo e separando o dataset.\n");
    load_and_split_data(INPUT_FILE, &train_data, &train_count, &valid_data, &valid_count);
    if(train_count == 0 || valid_count == 0){
        fprintf(stderr, "ERRO: Conjunto de treino ou validação está vazio. Verifique o arquivo de entrada e a lógica de separação.\n");
        return 1;
    }
    printf("  -> Treinamento: %ld registros | Validação: %ld registros\n\n", train_count, valid_count);

    printf("Calculando o modelo estatístico.\n");
    NumericalStats num_stats;
    CategoricalStats cat_stats[NUM_CATEGORICAL_COLS] = {0};
    calculate_stats_from_train(train_data, train_count, &num_stats, cat_stats);
    printf("  -> Modelo calculado com sucesso.\n\n");

    printf("Gerando %ld eventos sintéticos com simulação paralela.\n", valid_count);
    sim_data = malloc(valid_count * sizeof(DisasterRecord));
    run_simulation(valid_count, &num_stats, cat_stats, sim_data);
    printf("  -> Simulação concluída.\n\n");

    printf("Comparando estatísticas: Validação Real vs. Dados Simulados.\n");
    compare_and_print_stats(valid_data, valid_count, sim_data, valid_count);
    printf("\n");

    printf("Salvando dados simulados em '%s'.\n", OUTPUT_FILE);
    save_results(OUTPUT_FILE, sim_data, valid_count);
    printf("  -> Arquivo salvo.\n\n");

    cleanup(train_data, valid_data, sim_data, cat_stats);

    double end_time = omp_get_wtime();
    printf("Processo concluído com sucesso em %.4f segundos.\n", end_time - start_time);

    return 0;
}


void load_and_split_data(const char* filename, DisasterRecord** train_data, long* train_count, DisasterRecord** valid_data, long* valid_count) {
    FILE* fp = fopen(filename, "r");
    if (!fp) {
        fprintf(stderr, "ERRO: Não foi possível abrir o arquivo '%s'.\n", filename);
        exit(1);
    }

    char line[MAX_LINE_LEN];
    fgets(line, MAX_LINE_LEN, fp);

    long train_cap = 10000, valid_cap = 10000;
    *train_data = malloc(train_cap * sizeof(DisasterRecord));
    *valid_data = malloc(valid_cap * sizeof(DisasterRecord));
    *train_count = 0;
    *valid_count = 0;

    while (fgets(line, MAX_LINE_LEN, fp)) {
        DisasterRecord rec = {0};
        char* rest = line;
        char* token;
        int col_idx = 0;

        int start_impact_col = 7;
        int end_impact_col = 22;

        while ((token = strtok_r(rest, ",\n", &rest))) {
            if (token[0] == '"') {
                memmove(token, token + 1, strlen(token));
                if (token[strlen(token) - 1] == '"') {
                    token[strlen(token) - 1] = '\0';
                }
            }

            if (col_idx == 0) rec.ano = atoi(token);
            else if (col_idx == 1) strncpy(rec.uf, token, 3);
            else if (col_idx == 2) strncpy(rec.cobrade, token, 7);
            else if (col_idx >= start_impact_col && col_idx <= end_impact_col) {
                int impact_idx = col_idx - start_impact_col;
                rec.numerical_data[impact_idx] = atof(token);
            }
            col_idx++;
        }
        rec.uf[3] = '\0';
        rec.cobrade[7] = '\0';

        if (rec.ano > 0 && rec.ano < 2020) {
            if (*train_count >= train_cap) {
                train_cap *= 2;
                *train_data = realloc(*train_data, train_cap * sizeof(DisasterRecord));
            }
            (*train_data)[(*train_count)++] = rec;
        } else if (rec.ano >= 2020) {
             if (*valid_count >= valid_cap) {
                valid_cap *= 2;
                *valid_data = realloc(*valid_data, valid_cap * sizeof(DisasterRecord));
            }
            (*valid_data)[(*valid_count)++] = rec;
        }
    }
    fclose(fp);
}

void update_category(CategoricalStats* stat, const char* category_str, int** counts) {
    int found = 0;
    for (int i = 0; i < stat->count; i++) {
        if (strcmp(category_str, stat->categories[i]) == 0) {
            (*counts)[i]++;
            found = 1;
            break;
        }
    }
    if (!found) {
        if (stat->count >= stat->capacity) {
            stat->capacity = (stat->capacity == 0) ? 16 : stat->capacity * 2;
            stat->categories = realloc(stat->categories, stat->capacity * sizeof(char*));
            *counts = realloc(*counts, stat->capacity * sizeof(int));
        }
        stat->categories[stat->count] = strdup(category_str);
        (*counts)[stat->count] = 1;
        stat->count++;
    }
}

void calculate_stats_from_train(const DisasterRecord* train_data, long train_count, NumericalStats* num_stats, CategoricalStats cat_stats[NUM_CATEGORICAL_COLS]) {
    double sums[NUM_NUMERICAL_COLS] = {0};
    double sum_sqs[NUM_NUMERICAL_COLS] = {0};

    int* uf_counts = NULL;
    int* cobrade_counts = NULL;

    for (long i = 0; i < train_count; i++) {
        for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
            sums[j] += train_data[i].numerical_data[j];
            sum_sqs[j] += train_data[i].numerical_data[j] * train_data[i].numerical_data[j];
        }
        update_category(&cat_stats[0], train_data[i].uf, &uf_counts);
        update_category(&cat_stats[1], train_data[i].cobrade, &cobrade_counts);
    }

    for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
        num_stats->means[j] = sums[j] / train_count;
        double variance = sum_sqs[j] / train_count - num_stats->means[j] * num_stats->means[j];
        num_stats->std_devs[j] = (variance > 0) ? sqrt(variance) : 0.0;
    }

    cat_stats[0].probabilities = malloc(cat_stats[0].count * sizeof(double));
    for (int j = 0; j < cat_stats[0].count; j++) {
        cat_stats[0].probabilities[j] = (double)uf_counts[j] / train_count;
    }
    cat_stats[1].probabilities = malloc(cat_stats[1].count * sizeof(double));
    for (int j = 0; j < cat_stats[1].count; j++) {
        cat_stats[1].probabilities[j] = (double)cobrade_counts[j] / train_count;
    }

    free(uf_counts);
    free(cobrade_counts);
}


void run_simulation(long n_events, const NumericalStats* num_stats, const CategoricalStats cat_stats[NUM_CATEGORICAL_COLS], DisasterRecord* sim_data) {
    #pragma omp parallel
    {
        unsigned int seed = time(NULL) ^ omp_get_thread_num();

        #pragma omp for
        for (long i = 0; i < n_events; i++) {
            double p_uf = (double)rand_r(&seed) / RAND_MAX;
            double cumulative_p_uf = 0.0;
            int uf_idx = cat_stats[0].count - 1;
            for (int j = 0; j < cat_stats[0].count; j++) {
                cumulative_p_uf += cat_stats[0].probabilities[j];
                if (p_uf < cumulative_p_uf) {
                    uf_idx = j;
                    break;
                }
            }
            strcpy(sim_data[i].uf, cat_stats[0].categories[uf_idx]);

            double p_cobrade = (double)rand_r(&seed) / RAND_MAX;
            double cumulative_p_cobrade = 0.0;
            int cobrade_idx = cat_stats[1].count - 1;
            for (int j = 0; j < cat_stats[1].count; j++) {
                cumulative_p_cobrade += cat_stats[1].probabilities[j];
                if (p_cobrade < cumulative_p_cobrade) {
                    cobrade_idx = j;
                    break;
                }
            }
            strcpy(sim_data[i].cobrade, cat_stats[1].categories[cobrade_idx]);

            for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
                double sample = generate_normal_sample(num_stats->means[j], num_stats->std_devs[j], &seed);
                sim_data[i].numerical_data[j] = (sample > 0) ? sample : 0.0;
            }
        }
    }
}


void compare_and_print_stats(const DisasterRecord* valid_data, long valid_count, const DisasterRecord* sim_data, long sim_count) {
    NumericalStats valid_stats, sim_stats;
    double sums_valid[NUM_NUMERICAL_COLS] = {0}, sum_sqs_valid[NUM_NUMERICAL_COLS] = {0};
    double sums_sim[NUM_NUMERICAL_COLS] = {0}, sum_sqs_sim[NUM_NUMERICAL_COLS] = {0};

    for (long i = 0; i < valid_count; i++) {
        for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
            sums_valid[j] += valid_data[i].numerical_data[j];
            sum_sqs_valid[j] += valid_data[i].numerical_data[j] * valid_data[i].numerical_data[j];
        }
    }
    for (long i = 0; i < sim_count; i++) {
        for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
            sums_sim[j] += sim_data[i].numerical_data[j];
            sum_sqs_sim[j] += sim_data[i].numerical_data[j] * sim_data[i].numerical_data[j];
        }
    }

    for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
        valid_stats.means[j] = sums_valid[j] / valid_count;
        double variance_v = sum_sqs_valid[j] / valid_count - valid_stats.means[j] * valid_stats.means[j];
        valid_stats.std_devs[j] = (variance_v > 0) ? sqrt(variance_v) : 0.0;

        sim_stats.means[j] = sums_sim[j] / sim_count;
        double variance_s = sum_sqs_sim[j] / sim_count - sim_stats.means[j] * sim_stats.means[j];
        sim_stats.std_devs[j] = (variance_s > 0) ? sqrt(variance_s) : 0.0;
    }

    const char* col_names[NUM_NUMERICAL_COLS] = {"Mortos", "Feridos", "Desabrigados", "Desalojados", "Hab. Danif.", "Hab. Destr.",
                                                 "Saude Danif.", "Saude Destr.", "Ensino Danif.", "Ensino Destr.", "Comun. Danif.",
                                                 "Comun. Destr.", "Infra Danif.", "Infra Destr.", "Prej. Publico", "Prej. Privado"};

    printf("----------------------------------------------------------------------------------\n");
    printf("%-18s | %-25s | %-25s\n", "Metrica", "Validacao Real", "Simulacao Monte Carlo");
    printf("%-18s | %-12s %-12s | %-12s %-12s\n", "", "Media", "Desvio Padrao", "Media", "Desvio Padrao");
    printf("----------------------------------------------------------------------------------\n");
    for (int j = 0; j < NUM_NUMERICAL_COLS; j++) {
        printf("%-18s | %-12.2f %-12.2f | %-12.2f %-12.2f\n",
               col_names[j],
               valid_stats.means[j], valid_stats.std_devs[j],
               sim_stats.means[j], sim_stats.std_devs[j]);
    }
    printf("----------------------------------------------------------------------------------\n");
}


void save_results(const char* filename, const DisasterRecord* sim_data, long sim_count) {
    FILE* fp = fopen(filename, "w");
    if (!fp) {
        perror("ERRO ao salvar o arquivo de resultados");
        return;
    }
    fprintf(fp, "Sigla_UF,Cod_Cobrade,DH_MORTOS,DH_FERIDOS,DH_DESABRIGADOS,DH_DESALOJADOS,DM_Hab_Danificadas,DM_Hab_Destruidas,DM_Saude_Danificadas,DM_Saude_Destruidas,DM_Ensino_Danificadas,DM_Ensino_Destruidas,DM_Comun_Danificadas,DM_Comun_Destruidas,DM_Infra_Danificadas,DM_Infra_Destruidas,PE_Publico,PE_Privado\n");
    for (long i = 0; i < sim_count; i++) {
        fprintf(fp, "%s,%s", sim_data[i].uf, sim_data[i].cobrade);
        for(int j=0; j<NUM_NUMERICAL_COLS; j++){
            const char* format = (j < NUM_NUMERICAL_COLS - 2) ? ",%.0f" : ",%.2f";
            fprintf(fp, format, sim_data[i].numerical_data[j]);
        }
        fprintf(fp, "\n");
    }
    fclose(fp);
}

void cleanup(DisasterRecord* train_data, DisasterRecord* valid_data, DisasterRecord* sim_data, CategoricalStats cat_stats[NUM_CATEGORICAL_COLS]) {
    free(train_data);
    free(valid_data);
    free(sim_data);
    for (int i = 0; i < 2; i++) {
        for (int j = 0; j < cat_stats[i].count; j++) {
            free(cat_stats[i].categories[j]);
        }
        free(cat_stats[i].categories);
        free(cat_stats[i].probabilities);
    }
}

double generate_normal_sample(double mean, double std_dev, unsigned int* seed) {
    double u1, u2, z0;
    do { u1 = (double)rand_r(seed) / RAND_MAX; } while (u1 == 0.0);
    u2 = (double)rand_r(seed) / RAND_MAX;
    z0 = sqrt(-2.0 * log(u1)) * cos(2.0 * M_PI * u2);
    return z0 * std_dev + mean;
}

Overwriting monte_carlo_omp.c


In [43]:
!gcc -o monte_carlo_run -fopenmp monte_carlo_omp.c -lm
!./monte_carlo_run

Iniciando processo de simulação de Monte Carlo.

Lendo e separando o dataset.
  -> Treinamento: 9576 registros | Validação: 124 registros

Calculando o modelo estatístico.
  -> Modelo calculado com sucesso.

Gerando 124 eventos sintéticos com simulação paralela.
  -> Simulação concluída.

Comparando estatísticas: Validação Real vs. Dados Simulados.
----------------------------------------------------------------------------------
Metrica            | Validacao Real            | Simulacao Monte Carlo    
                   | Media        Desvio Padrao | Media        Desvio Padrao
----------------------------------------------------------------------------------
Mortos             | 52928.81     581337.62    | 111388.29    176932.86   
Feridos            | 472372.60    5226134.85   | inf          0.00        
Desabrigados       | 773.84       2634.86      | 56651.29     84972.30    
Desalojados        | 513080.45    5253053.82   | 68630.15     113605.79   
Hab. Danif.        | 46.98     